### Step 1: Check the GPU


In [ ]:
!nvidia-smi

### Step 2: Install `uv` and Python 3.12


In [ ]:
!pip install -q uv
!uv python install 3.12

### Step 3: Clone the repository


In [ ]:
import os

REPO_URL = "https://github.com/luatducvo/3D-scene.git"
if not os.path.exists("/content/3D-scene"):
    !git clone $REPO_URL /content/3D-scene

%cd /content/3D-scene

### Step 4: Install `scene-compute` dependencies
Use `--extra gpu` for Depth Anything 3, SAM 3, GeoCalib and MetaCLIP 2.


In [ ]:
# Install GPU Algorithm dependencies.
!uv sync --python 3.12 --package scene-compute --extra gpu

### Step 5: Cache model weights on Google Drive
Add `HF_TOKEN` to Colab secrets and grant this notebook access for the gated SAM 3 weights.


In [ ]:
import os
from pathlib import Path

import yaml
from google.colab import drive, userdata
from huggingface_hub import HfApi

drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/model-weights"
MODELS_DIR = f"{DRIVE_DIR}/models"
os.makedirs(MODELS_DIR, exist_ok=True)

# Keep disposable Scene data on the runtime SSD and model weights on Drive.
os.environ["DATA_DIR"] = "/content/data"
os.environ["MODELS_DIR"] = MODELS_DIR
os.environ["HF_HOME"] = f"{MODELS_DIR}/hf"
os.environ["TORCH_HOME"] = f"{MODELS_DIR}/torch"
os.environ["MODELS_PIPELINE_CONFIG"] = "/content/3D-scene/backend/config/pipeline.yaml"
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

# Report which weights are already complete on Drive; download_models skips those.
# Mirrors download_weights() in backend/packages/scene-stages/src/scene_stages/gpu.py.
hf_api = HfApi(token=os.environ["HF_TOKEN"])


def report_hf(repo_id, filenames=None):
    info = hf_api.model_info(repo_id, files_metadata=True)
    snapshot = Path(os.environ["HF_HOME"]) / "hub" / f"models--{repo_id.replace('/', '--')}" / "snapshots" / info.sha
    files = [s for s in info.siblings if filenames is None or s.rfilename in filenames]
    missing = [
        s.rfilename
        for s in files
        if not (snapshot / s.rfilename).is_file()
        or (s.size is not None and (snapshot / s.rfilename).stat().st_size != s.size)
    ]
    if missing:
        print(f"download {repo_id}: {len(missing)} of {len(files)} files missing ({', '.join(missing)})")
    else:
        size_gb = sum(s.size or 0 for s in files) / 1e9
        print(f"cached   {repo_id}: {len(files)} files, {size_gb:.1f} GB")


stages = yaml.safe_load(Path(os.environ["MODELS_PIPELINE_CONFIG"]).read_text())["stages"]
report_hf(stages["recon"]["params"]["model_id"])
report_hf("facebook/sam3", ["sam3.pt"])
report_hf(stages["embed"]["params"]["model_id"])
geocalib = Path(os.environ["TORCH_HOME"]) / "hub" / "geocalib" / "pinhole.tar"
print(f"{'cached  ' if geocalib.is_file() else 'download'} GeoCalib pinhole: {geocalib}")

!uv run --python 3.12 python -m scene_compute.download_models

# Drive uploads new files in the background; a runtime reset before it finishes loses them.
# Wait for the upload, then mount again for the compute server.
drive.flush_and_unmount()
drive.mount("/content/drive")

### Step 6: Start the Cloudflare Tunnel
Copy the public HTTPS URL into `COLAB_URL` on your local machine.


In [ ]:
import re
import subprocess
import time

# Install cloudflared and forward the compute server on port 9000.
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
!pkill cloudflared || true
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:9001"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
for _ in range(40):
    line = tunnel_proc.stdout.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break
    time.sleep(0.5)

print("Public compute URL:", public_url)

### Step 7: Start the compute server
Add `COMPUTE_TOKEN` to Colab secrets and use the same value locally. Restarting this server clears Scene data while preserving model weights on Drive.


In [ ]:
# Reuse the model-cache configuration above; never print secrets.
os.environ["COMPUTE_TOKEN"] = userdata.get("COMPUTE_TOKEN")
os.environ["PORT"] = "9000"
os.environ["CHUNK_SIZE_MB"] = "100"
os.environ["REQUEST_TIMEOUT_S"] = "300"
os.environ["CLEAN_ON_START"] = "1"

print("Starting scene-compute on port 9001")
!uv run --python 3.12 uvicorn scene_compute.server:app --host 0.0.0.0 --port 9001